
# HTML, CSS et inspection d'une page web

**Site étudié :** [Books to Scrape](https://books.toscrape.com/)  
**Catégorie sélectionnée :** Nonfiction

## Étape 1 — Périmètre de collecte

| Indicateur | Valeur |
|---|---:|
| Catégorie | Nonfiction |
| Nombre de livres annoncé | 110 |
| Nombre de pages | 6 |
| Nombre maximal de livres par page | 20 |

La catégorie contient 110 livres répartis sur 6 pages.

## Étape 2 — Identification des fiches produits

Chaque livre est représenté par un élément HTML possédant la classe `product_pod`.

**Sélecteur CSS :** `.product_pod`

**Commande de vérification :**

```javascript
document.querySelectorAll('.product_pod').length
```

**Résultats :**

| Page | Nombre de fiches |
|---|---:|
| 1 | 20 |
| 2 | 20 |
| 3 | 20 |
| 4 | 20 |
| 5 | 20 |
| 6 | 10 |
| **Total** | **110** |

**Contrôle de cohérence :**

20 + 20 + 20 + 20 + 20 + 10 = **110 livres**

Le nombre de fiches identifiées correspond au nombre de livres annoncé par le site.

## Étape 3 — Identification des données à extraire

| Donnée | Sélecteur CSS | Source de la valeur |
|---|---|---|
| Titre | `a[title]` | Attribut `title` |
| Prix | `p.price_color` | Contenu texte |
| Disponibilité | `.in_stock` | Contenu texte |
| Note | `.star-rating` | Attribut `class` |
| Lien vers la fiche | `a[href]` | Attribut `href` |

**Remarque :** les sélecteurs permettent d'identifier les éléments HTML. L'extraction de leurs valeurs nécessite ensuite de lire leur contenu texte ou l'attribut correspondant.

## Étape 4 — Vérification des sélecteurs

Les sélecteurs suivants ont été testés dans la console du navigateur, sur la première page de la catégorie.

**Nombre de fiches produits :**

```javascript
document.querySelectorAll('.product_pod').length
// Résultat : 20
```

**Nombre de prix :**

```javascript
document.querySelectorAll('article.product_pod p.price_color').length
// Résultat : 20
```

**Nombre de notes :**

```javascript
document.querySelectorAll('article.product_pod .star-rating').length
// Résultat : 20
```

### Bilan des contrôles

| Élément | Nombre obtenu | Résultat |
|---|---:|---|
| Fiches produits | 20 | Conforme |
| Prix | 20 | Conforme |
| Notes | 20 | Conforme |

Les trois sélecteurs retournent le même nombre d'éléments.

Les contrôles de cardinalité sont cohérents avec les 20 livres affichés sur cette page.

## Étape 5 — Pagination

**Sélecteur du lien vers la page suivante :** `li.next a`

**Valeur de l'attribut `href` sur la première page :**

```text
page-2.html
```

Le lien vers la page suivante est relatif à l'URL de la page courante.

**Condition d'arrêt :**

Lorsque le lien vers la page suivante est absent, le script doit arrêter la pagination : il a atteint la dernière page.

L'absence de ce lien constitue une condition normale de fin de collecte.

## Étape 6 — Vérification du code source

Le prix d'un livre est présent dans le code source HTML de la page.

**Conséquence :**

La donnée peut être extraite directement à partir du HTML téléchargé, sans nécessiter l'exécution de JavaScript.

## Étape 7 — Informations supplémentaires

La fiche détaillée d'un livre contient des informations absentes de la liste des produits.

Deux informations ont été identifiées :

| Information | Emplacement |
|---|---|
| Taxe (`Tax`) | Tableau des informations produit |
| Quantité en stock (`Availability`) | Tableau des informations produit |

La liste des produits indique uniquement la disponibilité du livre, tandis que la fiche détaillée permet de connaître la quantité en stock.

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import clear_output

In [2]:
# Constantes de configuration du scraper
_URL = "https://books.toscrape.com/catalogue/category/books/nonfiction_13/page-1.html"
_URL_ACCUEIL = "https://books.toscrape.com/index.html"
_ENTETES = {"User-Agent": "bootcamp-data/1.0 (Wild Code School)"}
_ENCODING = "utf-8"
_NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

In [3]:
class ErreurHttp(Exception):
    """Exception levée lorsqu'une réponse HTTP présente un statut inattendu."""

    def __init__(self, code_erreur):
        super().__init__(f"Server returned status {code_erreur}")
        self.code_erreur = code_erreur

In [4]:
def lire_une_fiche(fiche):
    """Extrait le titre, le prix, la note et l'URL d'un livre.

    Args:
        fiche: Balise HTML représentant un livre du catalogue.

    Returns:
        dict: Informations extraites du livre.
    """

    dict_fiche = {}
    lien_titre = fiche.select_one("h3 a")
    rating = fiche.select_one("p.star-rating")["class"]

    dict_fiche["prix"] = fiche.select_one("p.price_color").get_text(strip=True)
    dict_fiche["titre"] = lien_titre["title"]
    dict_fiche["url"] = lien_titre["href"]
    dict_fiche["note"] = _NOTES[rating[1]]

    return dict_fiche


def recuperer_une_page(soup):
    """Extrait les livres d'une page du catalogue.

    Args:
        soup: Objet BeautifulSoup contenant la page HTML.

    Returns:
        pd.DataFrame: Tableau des livres présents sur la page.
    """

    fiches_livres = soup.select("article.product_pod")

    liste_fiche = []
    for fiche in fiches_livres:
        liste_fiche.append(lire_une_fiche(fiche))

    df = pd.DataFrame(liste_fiche)

    return df


def extraire_nombre_livres(soup):
    """Extrait le nombre total de livres annoncé sur une page de catégorie.

    Args:
        soup (BeautifulSoup): Contenu HTML de la page à analyser.

    Returns:
        int: Nombre total de livres annoncé par le site.
    """

    nombre_str = soup.select_one("form.form-horizontal strong").get_text(
        strip=True
    )
    nombre = int(nombre_str)
    return nombre

In [ ]:
def recuperer_soup(url):
    """Récupère une page HTML et la convertit en objet BeautifulSoup.

    Args:
        url: URL de la page à récupérer.

    Returns:
        BeautifulSoup: Contenu HTML analysé.

    Raises:
        ErreurHttp: Si le serveur retourne un statut HTTP différent de 200.
    """

    request = requests.get(url, headers=_ENTETES, timeout=10)
    request.encoding = _ENCODING

    if request.status_code != 200:
        raise ErreurHttp(request.status_code)
    soup = BeautifulSoup(request.text, "html.parser")

    return soup


def collecter_toutes_pages(url_depart, pages_max=10):
    """Collecte les livres d'une catégorie sur plusieurs pages.

    Args:
        url_depart: URL de la première page du catalogue.
        pages_max: Nombre maximal de pages à parcourir (10 par défaut).

    Returns:
        pd.DataFrame: Livres collectés avec leurs URL absolues.
        None: Si aucune page n'a été parcourue.
    """

    url = url_depart
    df_livres = None
    page = 1

    while url and page <= pages_max:
        soup = recuperer_soup(url)

        df_recuperé = recuperer_une_page(soup)

        # Résout les liens relatifs depuis l'URL de la page courante.
        df_recuperé["url"] = df_recuperé["url"].apply(
            lambda x: urljoin(url, x) # noqa: B023
        )

        df_livres = pd.concat([df_livres, df_recuperé], ignore_index=True)
        print(
            f"Page {page} : {df_recuperé.shape[0]} livres "
            f"(total {df_livres.shape[0]})"
        )

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        page += 1
        time.sleep(1)

    return df_livres

In [6]:
def recuperer_stock(availability):
    """Extrait la quantité disponible à partir du texte de disponibilité.

    Args:
        availability: Texte indiquant la disponibilité du produit.

    Returns:
        int: Quantité disponible, ou 0 si le format n'est pas reconnu.
        None: Si le texte de disponibilité est absent.
    """

    if not availability:
        return None

    trouve = re.search(r"\((\d+) available\)", availability)
    return int(trouve.group(1)) if trouve else 0


def fiche_detaillee(soup):
    """Extrait les informations détaillées d'un livre.

    Args:
        soup: Objet BeautifulSoup contenant la fiche du livre.

    Returns:
        dict: UPC, stock disponible et catégorie du livre.
    """

    informations_produit = {}
    # Convertit le tableau HTML des caractéristiques en dictionnaire.
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        valeur = ligne.select_one("td")
        if cle and valeur:
            informations_produit[cle.get_text(strip=True)] = valeur.get_text(
                strip=True
            )

    fil_ariane = soup.select("ul.breadcrumb li a")
    categorie = fil_ariane[-1].get_text(strip=True) if fil_ariane else None

    details_livre = {}
    details_livre["upc"] = informations_produit.get("UPC")
    details_livre["stock"] = recuperer_stock(
        informations_produit.get("Availability")
    )
    details_livre["categorie"] = categorie

    return details_livre


def recuperer_details(df):
    """Récupère les informations détaillées de chaque livre du DataFrame.

    Args:
        df: DataFrame contenant les titres et les URL absolues des livres.

    Returns:
        list[dict]: Informations détaillées associées aux URL des livres.
    """

    details = []
    for row in df.itertuples():
        soup = recuperer_soup(row.url)
        details_livre = fiche_detaillee(soup)
        details_livre["url"] = row.url
        details.append(details_livre)
        print(f"Fiche détaillée de {row.titre} traitée")
        time.sleep(1)

    return details

In [ ]:
def construire_dataframe_final(df_catalogue, details_livres):
    """Construit le DataFrame final à partir du catalogue et des fiches détaillées.

    Fusionne les données sur les URL, convertit les prix en livres sterling
    en valeurs numériques et sélectionne les colonnes finales.

    Args:
        df_catalogue (pd.DataFrame): Données extraites du catalogue.
        details_livres (list[dict]): Données extraites des fiches détaillées.

    Returns:
        pd.DataFrame: Données fusionnées, nettoyées et structurées.
    """

    df2 = pd.DataFrame(details_livres)
    df_combine = pd.merge(
        df_catalogue, df2, how="inner", left_on="url", right_on="url"
    )

    # Convertit les prix en valeurs numériques, exprimées en livres sterling.
    df_combine["prix"] = df_combine["prix"].str.replace("£", "")
    df_combine["prix"] = df_combine["prix"].astype(float)
    df_combine = df_combine.rename(columns={"prix": "prix_gbp"})

    df_combine = df_combine[
        ["titre", "categorie", "prix_gbp", "note", "stock", "upc", "url"]
    ]

    return df_combine

In [8]:
def verifier_integrite_donnees(df_final, df_catalogue, details_livres):
    """Vérifie l'intégrité des données après la collecte et la jointure.

    Contrôle les cardinalités, la correspondance et l'unicité des URL,
    ainsi que l'absence de valeurs manquantes dans le DataFrame final.

    Args:
        df_final (pd.DataFrame): Données fusionnées et structurées.
        df_catalogue (pd.DataFrame): Données extraites du catalogue.
        details_livres (list[dict]): Données extraites des fiches détaillées.

    Returns:
        None: Affiche le résultat des six contrôles d'intégrité.
    """

    url_livres = set(df_catalogue["url"])
    url_details = {x["url"] for x in details_livres}

    print(
        "1. Cardinalité des collectes :",
        len(details_livres) == df_catalogue.shape[0],
    )

    print("2. Correspondance des URL :", url_livres == url_details)

    print(
        "3. Unicité des URL des livres :",
        len(url_livres) == df_catalogue.shape[0],
    )

    print(
        "4. Unicité des URL des fiches détaillées :",
        len(url_details) == len(details_livres),
    )

    print(
        "5. Cardinalité après jointure :",
        df_final.shape[0] == df_catalogue.shape[0],
    )

    print(
        "6. Absence de valeurs manquantes :", df_final.isna().sum().sum() == 0
    )

In [9]:
def afficher_analyse_descriptive(df_livres):
    """Calcule et affiche les statistiques descriptives des livres collectés.

    Présente le nombre de livres, le stock total, le prix moyen, le nombre
    de catégories, le livre le plus cher et la répartition des notes.

    Args:
        df_livres (pd.DataFrame): Données des livres collectés et validés.

    Returns:
        None: Affiche les résultats de l'analyse descriptive.
    """

    nombre_livres = df_livres.shape[0]
    stock_total = df_livres["stock"].sum()
    prix_moyen = df_livres["prix_gbp"].mean()
    nombre_categories = df_livres["categorie"].nunique()
    print("========== ANALYSE DES LIVRES ==========\n")

    print(f"Nombre de livres : {nombre_livres}")
    print(f"Stock total (exemplaires) : {stock_total}")
    print(f"Prix moyen : {prix_moyen:.2f} £")
    print(f"Nombre de catégories distinctes : {nombre_categories}")
    print("-" * 40)
    print("--- Livre le plus cher ---")
    print(df_livres.nlargest(1, "prix_gbp").iloc[0])

    print("-" * 40)
    print("--- Répartition des notes (%) ---")
    print(round(df_livres.groupby("note").size() / nombre_livres * 100.0, 2))

    print("\n========== FIN DE L'ANALYSE ==========")

Il n'y a qu'une seule catégorie différente dans le dataframe, la collecte est donc cohérente avec le périmètre prévu

In [10]:
def exporter_csv(df_livres):
    """Exporte les données collectées dans un fichier CSV.

    Nomme le fichier selon la catégorie des livres et le place
    dans le dossier data, créé automatiquement si nécessaire.

    Args:
        df_livres (pd.DataFrame): Données des livres à exporter.

    Returns:
        None: Enregistre le fichier CSV sur le disque.
    """

    nom_fichier = df_livres.iloc[0]["categorie"] + ".csv"
    dossier_data = Path.cwd().parent / "data"
    dossier_data.mkdir(parents=True, exist_ok=True)
    chemin_csv = dossier_data / nom_fichier
    df_livres.to_csv(chemin_csv, index=False)

In [ ]:
def extraire_categories(soup, url):
    """Extrait les catégories de livres et leurs URL depuis une page HTML.

    Args:
        soup (BeautifulSoup): Contenu HTML de la page à analyser.
        url (str): URL de référence pour résoudre les liens relatifs.

    Returns:
        list[dict]: Catégories contenant chacune un nom et une URL absolue.
    """

    liens = soup.select("ul.nav-list ul > li > a")

    categories = []
    for lien in liens:
        nom_categorie = lien.get_text(strip=True)
        url_categorie = urljoin(url, lien["href"])

        categories.append({"categorie": nom_categorie, "url": url_categorie})

    return categories


def selectionner_categorie(categories):
    """Affiche les catégories disponibles et demande à l'utilisateur d'en choisir une.

    Vérifie que la saisie est un entier compris dans les limites de la liste
    et redemande un choix tant que la saisie est invalide.

    Args:
        categories (list[dict]): Catégories contenant chacune un nom et une URL.

    Returns:
        dict: Catégorie sélectionnée, avec son nom et son URL.
    """

    for numero, categorie in enumerate(categories, start=1):
        print(f"{numero} {categorie['categorie']}")

    while True:
        try:
            saisie = input(
                f"Rentrez le numéro de la catégorie [1, {len(categories)}] : "
            )
            choix = int(saisie)
            if choix < 1 or choix > len(categories):
                print(
                    f"Le numéro doit être compris entre 1 et {len(categories)} !"
                )
                continue
            break
        except ValueError:
            print(f"'{saisie}' n'est pas un numéro valide !")
            continue

    return categories[choix - 1]

In [ ]:
def orchestrateur(url):
    """Orchestre la collecte, la validation et l'export des livres d'une catégorie.

    Récupère les livres du catalogue et leurs fiches détaillées, construit
    le DataFrame final, vérifie son intégrité, affiche les statistiques
    descriptives et exporte les données au format CSV.

    Args:
        url (str): URL de la première page de la catégorie à collecter.

    Returns:
        None: Affiche les résultats et exporte les données dans un fichier CSV.
    """

    df_livres = collecter_toutes_pages(url, 12)

    # Vérifie le nombre attendu de livres.
    nombre_attendu = extraire_nombre_livres(recuperer_soup(url))
    print(
        f"{df_livres.shape[0]} (attendu {nombre_attendu}) : {df_livres.shape[0] == nombre_attendu}"
    )

    liste_details_livres = recuperer_details(df_livres)
    df_total = construire_dataframe_final(df_livres, liste_details_livres)
    verifier_integrite_donnees(df_total, df_livres, liste_details_livres)
    afficher_analyse_descriptive(df_total)
    exporter_csv(df_total)

In [16]:
# Version Nonfiction de test
orchestrateur(_URL)

Page 1 : 20 livres (total 20)
Page 2 : 20 livres (total 40)
Page 3 : 20 livres (total 60)
Page 4 : 20 livres (total 80)
Page 5 : 20 livres (total 100)
Page 6 : 10 livres (total 110)
110 (attendu 110) : True
Fiche détaillée de Worlds Elsewhere: Journeys Around Shakespeare’s Globe traitée
Fiche détaillée de The Five Love Languages: How to Express Heartfelt Commitment to Your Mate traitée
Fiche détaillée de Reasons to Stay Alive traitée
Fiche détaillée de #HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe. traitée
Fiche détaillée de Unseen City: The Majesty of Pigeons, the Discreet Charm of Snails & Other Wonders of the Urban Wilderness traitée
Fiche détaillée de Throwing Rocks at the Google Bus: How Growth Became the Enemy of Prosperity traitée
Fiche détaillée de The Life-Changing Magic of Tidying Up: The Japanese Art of Decluttering and Organizing traitée
Fiche détaillée de The Gutsy Girl: Escapades for Your Life of Epic Adventure traitée
Fiche détaillée de The Electric P

In [15]:
# Version choix catégorie
categories = extraire_categories(recuperer_soup(_URL_ACCUEIL), _URL_ACCUEIL)
choix = selectionner_categorie(categories)
clear_output(wait=True)
orchestrateur(choix["url"])

Page 1 : 11 livres (total 11)
11 (attendu 11) : True
Fiche détaillée de It's Only the Himalayas traitée
Fiche détaillée de Full Moon over Noah’s Ark: An Odyssey to Mount Ararat and Beyond traitée
Fiche détaillée de See America: A Celebration of Our National Parks & Treasured Sites traitée
Fiche détaillée de Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel traitée
Fiche détaillée de Under the Tuscan Sun traitée
Fiche détaillée de A Summer In Europe traitée
Fiche détaillée de The Great Railway Bazaar traitée
Fiche détaillée de A Year in Provence (Provence #1) traitée
Fiche détaillée de The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2) traitée
Fiche détaillée de Neither Here nor There: Travels in Europe traitée
Fiche détaillée de 1,000 Places to See Before You Die traitée
1. Cardinalité des collectes : True
2. Correspondance des URL : True
3. Unicité des URL des livres : True
4. Unicité des URL des fiches détaillées : True